## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-05-19-54-45 +0000,chitri,Orland Park working to fix AI-powered call sys...,https://www.chicagotribune.com/2026/11/05/orla...
1,2026-11-05-19-52-00 +0000,wsj,An unusual force helped fuel this week’s histo...,https://www.wsj.com/politics/elections/the-cos...
2,2026-11-05-19-51-15 +0000,nypost,"FDNY commissioner 1,600 miles away at Puerto R...",https://nypost.com/2026/11/05/us-news/fdny-com...
3,2026-11-05-19-50-59 +0000,bbc,California governor commutes death sentences f...,https://www.bbc.co.uk/news/articles/cqrkyyd2j8...
4,2026-11-05-19-50-57 +0000,nypost,Married couple drown in river after their infl...,https://nypost.com/2026/11/05/us-news/married-...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2467/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
297,trump,46
65,democrats,28
16,midterm,27
8,new,19
349,midterms,16


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
425,2026-11-04-20-06-00 +0000,wsj,"Democrats won control of Congress, a majority ...",https://www.wsj.com/politics/elections/how-dem...,171
398,2026-11-04-21-21-00 +0000,wsj,Markets surged Wednesday after American voters...,https://www.wsj.com/politics/elections/the-win...,145
369,2026-11-04-22-37-00 +0000,wsj,Democrats’ midterm sweep has shaken up the 202...,https://www.wsj.com/politics/elections/democra...,142
99,2026-11-05-17-23-49 +0000,bbc,Farage says he is worried Trump will step up I...,https://www.bbc.co.uk/news/articles/cqz0vz55jp...,140
102,2026-11-05-17-20-00 +0000,wsj,President Trump put three justices on the Supr...,https://www.wsj.com/politics/policy/midterm-ro...,138


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
425,171,2026-11-04-20-06-00 +0000,wsj,"Democrats won control of Congress, a majority ...",https://www.wsj.com/politics/elections/how-dem...
186,99,2026-11-05-13-26-00 +0000,wsj,The Bank of England raised its key interest ra...,https://www.wsj.com/economy/central-banking/ba...
369,87,2026-11-04-22-37-00 +0000,wsj,Democrats’ midterm sweep has shaken up the 202...,https://www.wsj.com/politics/elections/democra...
168,74,2026-11-05-14-23-16 +0000,nypost,Russell Brand convicted of rape in UK court — ...,https://nypost.com/2026/11/05/world-news/russe...
286,70,2026-11-05-06-13-59 +0000,nypost,Ted Cruz calls midterm election results a ‘Wat...,https://nypost.com/2026/11/05/us-news/ted-cruz...
411,67,2026-11-04-20-45-47 +0000,nypost,Voters reject data centers in nearly 50 ballot...,https://nypost.com/2026/11/04/business/nearly-...
182,53,2026-11-05-13-43-28 +0000,chitri,Illinois House Speaker Chris Welch faces revol...,https://www.chicagotribune.com/2026/11/05/illi...
1,48,2026-11-05-19-52-00 +0000,wsj,An unusual force helped fuel this week’s histo...,https://www.wsj.com/politics/elections/the-cos...
54,48,2026-11-05-18-50-04 +0000,nypost,Spencer Pratt issues scathing takedown of Kare...,https://nypost.com/2026/11/05/us-news/spencer-...
3,48,2026-11-05-19-50-59 +0000,bbc,California governor commutes death sentences f...,https://www.bbc.co.uk/news/articles/cqrkyyd2j8...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
